# DeepOptimizedNN Training Pipeline

This notebook implements and explains the **DeepOptimizedNN** — a deep feedforward neural network for classification tasks.  
It includes model definition, training/evaluation pipeline, and in-depth commentary on each design choice.


In [3]:
# 1. Imports and Setup

import uuid
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import os
from machine_learning.models.utils import bar_plot
 

In [5]:
# 1. The Model: DeepOptimizedNN

class DeepOptimizedNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DeepOptimizedNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, output_dim)
        )

    def forward(self, x):
        return self.net(x)


### Architecture Overview

This is a fully connected **Deep Neural Network (DNN)** built using PyTorch’s `nn.Sequential`, stacking layers linearly.

| Layer | Type | Purpose |
|-------|------|----------|
| `nn.Linear(input_dim, 128)` | Dense Layer | Projects input features into a 128D latent space |
| `nn.BatchNorm1d(128)` | Batch Normalization | Stabilizes and accelerates training |
| `nn.LeakyReLU()` | Activation | Allows gradient flow for negative inputs |
| `nn.Dropout(0.3)` | Regularization | Randomly zeroes 30% activations to prevent overfitting |
| `nn.Linear(128, 64)` + BatchNorm/ReLU/Dropout | Hidden Layer | Learns higher-level patterns |
| `nn.Linear(64, 32)` + BatchNorm/ReLU | Compression | Extracts abstract features |
| `nn.Linear(32, 16)` + ReLU | Penultimate Layer | Reduces representation size |
| `nn.Linear(16, output_dim)` | Output Layer | Produces logits for classification |

**Design Rationale:**
- Gradual layer size reduction = feature hierarchy funnel  
- Mix of LeakyReLU and ReLU = robustness against dead neurons  
- BatchNorm + Dropout = faster convergence + regularization  
- Result: general-purpose deep classifier


## 4. Hyperparameter Explanations and Design Choices

### 1. epochs = 40
- Maximum number of full dataset passes.  
- Early stopping halts earlier if no improvement.  
- Too low → underfit; too high → overfit.

### 2. batch_size = 256
- Number of samples per optimizer step.  
- 256 offers stable gradients and efficient batchnorm stats.  

### 3. lr = 1e-3
- Step size of optimizer updates.  
- Standard for AdamW, balanced for convergence/stability.  

### 4. patience = 5
- Stops if validation loss doesn’t improve for 5 epochs.  
- Prevents overfitting and wasted computation.  

### 5. max_train_samples = 60000
- Caps data for speed/debugging.  
- Randomly samples subset if dataset is huge.  

### 6. Dropout layers = 0.3, 0.25
- Randomly zero out neurons → prevents co-adaptation.  
- Heavier dropout in larger layers.  

### 7. Weight Decay = 1e-4
- L2 regularization for smaller, smoother weights.  
- Helps generalization.  

### 8. Optimizer = AdamW
- Adaptive gradient descent with proper decoupled weight decay.  
- Fast, stable, and generalizes well.  

### 9. BatchNorm1d
- Normalizes activations per batch.  
- Stabilizes training and enables higher learning rates.  

### 10. Activation Functions
- **LeakyReLU** in first layer (prevents dead neurons).  
- **ReLU** later layers (efficient post-normalization).  

### 11. GradScaler & autocast
- Mixed precision training (FP16/FP32).  
- Reduces memory use and speeds up training on GPUs.  

### 12. Early Stopping + best_state
- Keeps best validation-loss model.  
- Avoids using overfitted weights.  

### 13. Learning Curve Visualization
- Plots training vs validation loss to inspect convergence/overfitting.  

### 14. classification_report & bar_plot
- `classification_report` → precision, recall, F1 per class.  
- `bar_plot` → visual per-class performance overview.  


## 15. Interactions and Trade-offs Summary

| Hyperparameter | Increase → | Decrease → |
|----------------|-------------|-------------|
| **epochs** | Better convergence, possible overfit | Faster training, possible underfit |
| **batch_size** | Smoother gradients, less generalization | Noisier gradients, possible better generalization |
| **lr** | Faster convergence, risk of divergence | More stable, slower learning |
| **dropout** | Less overfitting, slower learning | Faster learning, risk of overfitting |
| **patience** | More tolerance for fluctuation | May stop too early |
| **weight_decay** | Smoother model, lower overfit | More flexible, risk of overfit |
